# 03 - Stain Normalization & Deconvolution Comparison

## The Stain Normalization Problem

H&E staining produces images with significant color variation due to:
- Different stain concentrations across labs
- Scanner variability
- Tissue preparation differences

The paper (Section 3.4) evaluates **6 methods** to address this:

| # | Method | Type | Key Idea |
|---|--------|------|----------|
| 1 | CLAHE | Enhancement | Adaptive histogram equalization |
| 2 | Hoque et al. | Deconvolution | PCA-based stain vector estimation |
| 3 | Macenko et al. | Normalization | SVD-geodesic stain vector estimation |
| 4 | Khan et al. | Normalization | Color descriptor + random forest classifier |
| 5 | Alsubaie et al. | Deconvolution | ICA in wavelet domain |
| 6 | Zheng et al. | Deconvolution | Adaptive color deconvolution |

### Our POC implements:
- **CLAHE** (baseline)
- **Macenko** (best on gastric cancer dataset, F1=0.854)
- **Reinhard** (classic color transfer)
- **Vahadane** (sparse NMF-based)
- **Ruifrok** (standard color deconvolution)
- **PCA-based** deconvolution (similar to Hoque et al.)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import cv2
import warnings
warnings.filterwarnings('ignore')

from src.utils.data_loader import load_dataset
from src.utils.visualization import compare_normalizations, compare_deconvolutions
from src.preprocessing.stain_normalization import (
    normalize_clahe, normalize_macenko, normalize_reinhard, 
    normalize_vahadane_simple, normalize_image
)
from src.preprocessing.stain_deconvolution import (
    deconvolve_ruifrok, deconvolve_pca
)

plt.rcParams['figure.dpi'] = 100
DATA_ROOT = '../../data'

images_m, masks_m, names_m = load_dataset(DATA_ROOT, 'monuseg', max_images=4)
images_t, masks_t, names_t = load_dataset(DATA_ROOT, 'tnbc', max_images=4)
print(f'Loaded {len(images_m)} MoNuSeg + {len(images_t)} TNBC images')

## 1. Visual Comparison of Normalization Methods

Let's apply each normalization method and see how the images change.

In [ ]:
# Apply all normalization methods to a sample image
img = images_m[0]

# CLAHE
clahe_norm, clahe_h = normalize_clahe(img)

# Macenko
macenko_norm, macenko_h, macenko_e = normalize_macenko(img)

# Reinhard
reinhard_norm = normalize_reinhard(img)

# Vahadane
vahadane_norm, vahadane_h, vahadane_e = normalize_vahadane_simple(img)

normalized = {
    'CLAHE': clahe_norm,
    'Macenko': macenko_norm,
    'Reinhard': reinhard_norm,
    'Vahadane (NMF)': vahadane_norm,
}

fig = compare_normalizations(img, normalized, figsize=(25, 5))
plt.suptitle('Stain Normalization Methods Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("Each method produces different color profiles.")
print("Macenko and Vahadane normalize to a reference stain appearance.")
print("CLAHE enhances contrast but doesn't truly normalize stain colors.")

In [ ]:
# Apply to multiple images - show normalization consistency
fig, axes = plt.subplots(len(images_m), 5, figsize=(25, 5*len(images_m)))

for i, img in enumerate(images_m):
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f'Original: {names_m[i]}' if i == 0 else names_m[i])
    axes[i, 0].axis('off')
    
    methods = [
        ('CLAHE', normalize_clahe(img)[0]),
        ('Macenko', normalize_macenko(img)[0]),
        ('Reinhard', normalize_reinhard(img)),
        ('Vahadane', normalize_vahadane_simple(img)[0]),
    ]
    
    for j, (name, norm) in enumerate(methods, 1):
        axes[i, j].imshow(norm)
        if i == 0:
            axes[i, j].set_title(name)
        axes[i, j].axis('off')

plt.suptitle('Normalization Consistency Across Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNotice: After Macenko normalization, all images have more consistent coloring.")
print("This consistency is crucial for reliable automated analysis.")

## 2. Stain Deconvolution: Separating H and E

Color deconvolution separates the combined H&E stain into individual channels:
- **Hematoxylin channel**: Shows nuclei (our primary target)
- **Eosin channel**: Shows cytoplasm and stroma

In [ ]:
# Compare deconvolution methods
img = images_m[0]

# Ruifrok standard deconvolution
ruifrok_result = deconvolve_ruifrok(img)

# PCA-based deconvolution (similar to paper's Method 2)
pca_result = deconvolve_pca(img)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

# Original
axes[0, 0].imshow(img)
axes[0, 0].set_title('Original H&E Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')
axes[0, 1].axis('off')
axes[0, 2].axis('off')

# Ruifrok
axes[1, 0].imshow(ruifrok_result['normalized'])
axes[1, 0].set_title('Ruifrok - Reconstructed', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(ruifrok_result['hematoxylin'], cmap='gray')
axes[1, 1].set_title('Ruifrok - Hematoxylin', fontsize=11)
axes[1, 1].axis('off')

axes[1, 2].imshow(ruifrok_result['eosin'], cmap='gray')
axes[1, 2].set_title('Ruifrok - Eosin', fontsize=11)
axes[1, 2].axis('off')

# PCA
axes[2, 0].imshow(pca_result['normalized'])
axes[2, 0].set_title('PCA-based - Reconstructed', fontsize=11)
axes[2, 0].axis('off')

axes[2, 1].imshow(pca_result['hematoxylin'], cmap='gray')
axes[2, 1].set_title('PCA-based - Hematoxylin', fontsize=11)
axes[2, 1].axis('off')

axes[2, 2].imshow(pca_result['eosin'], cmap='gray')
axes[2, 2].set_title('PCA-based - Eosin', fontsize=11)
axes[2, 2].axis('off')

plt.suptitle('Stain Deconvolution: Separating Hematoxylin and Eosin', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Hematoxylin channel clearly highlights nuclei - this is what we segment.")
print("PCA-based method adapts to the specific image, while Ruifrok uses fixed vectors.")

## 3. Why Normalization Helps Segmentation

Let's compare segmentation with and without normalization.

In [ ]:
from src.segmentation.otsu_watershed import segment_otsu_watershed, segment_adaptive_watershed
from src.evaluation.metrics import compute_all_metrics

results = []

for i, (img, mask, name) in enumerate(zip(images_m, masks_m, names_m)):
    # 1. No normalization (baseline)
    seg_basic, _ = segment_otsu_watershed(img)
    m_basic = compute_all_metrics(mask, seg_basic)
    m_basic['Method'] = 'No Normalization'
    m_basic['Image'] = name
    results.append(m_basic)
    
    # 2. CLAHE
    clahe_n, clahe_h = normalize_clahe(img)
    seg_clahe, _ = segment_otsu_watershed(clahe_n)
    m_clahe = compute_all_metrics(mask, seg_clahe)
    m_clahe['Method'] = 'CLAHE'
    m_clahe['Image'] = name
    results.append(m_clahe)
    
    # 3. Macenko + Hematoxylin channel
    try:
        mac_n, mac_h, mac_e = normalize_macenko(img)
        seg_mac, _ = segment_otsu_watershed(mac_h)
        m_mac = compute_all_metrics(mask, seg_mac)
        m_mac['Method'] = 'Macenko'
        m_mac['Image'] = name
        results.append(m_mac)
    except Exception as e:
        print(f"  Macenko failed on {name}: {e}")
    
    # 4. Ruifrok deconvolution + Hematoxylin channel
    ruif = deconvolve_ruifrok(img)
    seg_ruif, _ = segment_otsu_watershed(ruif['hematoxylin'])
    m_ruif = compute_all_metrics(mask, seg_ruif)
    m_ruif['Method'] = 'Ruifrok Deconv'
    m_ruif['Image'] = name
    results.append(m_ruif)

import pandas as pd
df = pd.DataFrame(results)
print("\nPer-image metrics:")
print(df[['Image', 'Method', 'F1', 'Dice', 'AJI']].to_string(index=False))

In [ ]:
# Average metrics by method
avg_metrics = df.groupby('Method')[['F1', 'Dice', 'AJI', 'Accuracy']].mean()
print("\nAverage Metrics by Method:")
print(avg_metrics.round(3))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 6))
avg_metrics[['F1', 'Dice', 'AJI']].plot(kind='bar', ax=ax, width=0.7)
ax.set_title('Segmentation Metrics: Effect of Stain Normalization', 
             fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(loc='upper right')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)

plt.tight_layout()
plt.show()

print("\nPaper's key finding: Stain normalization improves F1 by up to 5.8%.")
print("Macenko achieves 0.854 F1 on gastric cancer dataset (Table 2 in paper).")

## 4. Color Space Analysis

Let's visualize how normalization affects the color distribution.

In [ ]:
# Show how normalization changes color distributions
img = images_m[0]
mac_n, _, _ = normalize_macenko(img)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
channel_names = ['Red', 'Green', 'Blue']
colors = ['red', 'green', 'blue']

for ch in range(3):
    # Before normalization
    for i, im in enumerate(images_m[:4]):
        axes[0, ch].hist(im[:,:,ch].ravel(), bins=50, alpha=0.3, 
                        color=colors[ch], density=True)
    axes[0, ch].set_title(f'{channel_names[ch]} - Before Normalization')
    axes[0, ch].set_xlabel('Pixel Value')
    
    # After Macenko normalization
    for i, im in enumerate(images_m[:4]):
        try:
            n, _, _ = normalize_macenko(im)
            axes[1, ch].hist(n[:,:,ch].ravel(), bins=50, alpha=0.3, 
                            color=colors[ch], density=True)
        except:
            pass
    axes[1, ch].set_title(f'{channel_names[ch]} - After Macenko')
    axes[1, ch].set_xlabel('Pixel Value')

plt.suptitle('Color Distributions: Before vs After Normalization\n(Each line = different image)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("After normalization, color distributions are more aligned across images.")
print("This is critical for consistent thresholding in segmentation.")

## Summary

### From the paper (Tables 2 & 3):

| Method | F1 (Gastric) | F1 (TCGA) |
|--------|-------------|----------|
| Basic (no normalization) | 0.807 | 0.847 |
| **Macenko** | **0.854** | 0.896 |
| **Hoque (CD)** | 0.847 | **0.907** |
| Khan | 0.831 | 0.875 |
| Alsubaie (ICA) | 0.797 | 0.837 |
| Zheng (ACD) | 0.804 | 0.888 |

**Key insight**: Up to 5.8% F1 improvement with normalization!

The choice of method matters - Macenko excels on gastric cancer, while
PCA-based deconvolution works better on multi-organ TCGA dataset.